# Qwen Blockwise-ControlNet Inpaint (Local Bundle)

이 노트북은 `/mnt/local/required-for-notebook` 아래에 업로드 번들을 받아둔 상태에서 **추가 다운로드 없이** 실행됩니다.

- models: `/mnt/local/required-for-notebook/models/...`
- dataset: `/mnt/local/required-for-notebook/dataset/...`
- output: `/mnt/local/required-for-notebook/output/...`


In [ ]:
from pathlib import Path
import json
import glob

import torch
from PIL import Image
from IPython.display import display

from diffsynth.pipelines.qwen_image import QwenImagePipeline, ModelConfig, ControlNetInput

BUNDLE_ROOT = Path("/mnt/local/required-for-notebook")
MODEL_ROOT = BUNDLE_ROOT / "models"
QWEN_ROOT = MODEL_ROOT / "Qwen" / "Qwen-Image"
CONTROLNET_DIR = MODEL_ROOT / "train" / "Qwen-Image-Blockwise-ControlNet-Inpaint_full"

DATASET_ROOT = BUNDLE_ROOT / "dataset"
VAL_JSONL_PATH = DATASET_ROOT / "val_blockwise_controlnet_inpaint.jsonl"

VAL_SAMPLE_INDEX = 0
NEGATIVE_PROMPT = ""
NUM_INFERENCE_STEPS = 40
CFG_SCALE = 4.0
SEED = 0
CUDA_DEVICE_INDEX = 0
MAX_LONG_SIDE = 1024

if not VAL_JSONL_PATH.exists():
    raise FileNotFoundError(f"Validation jsonl not found: {VAL_JSONL_PATH}")

with open(VAL_JSONL_PATH, "r", encoding="utf-8") as f:
    val_records = [json.loads(line) for line in f if line.strip()]

if not val_records:
    raise ValueError(f"No validation records found: {VAL_JSONL_PATH}")

VAL_SAMPLE_INDEX = max(0, min(VAL_SAMPLE_INDEX, len(val_records) - 1))
sample = val_records[VAL_SAMPLE_INDEX]

PROMPT = sample["prompt"]
INPUT_IMAGE_PATH = DATASET_ROOT / sample["blockwise_controlnet_image"]
MASK_IMAGE_PATH = DATASET_ROOT / sample["blockwise_controlnet_inpaint_mask"]
TARGET_IMAGE_PATH = DATASET_ROOT / sample["image"]

checkpoints = sorted(glob.glob(str(CONTROLNET_DIR / "step-*.safetensors")))
if not checkpoints:
    raise FileNotFoundError(f"No trained checkpoint found in: {CONTROLNET_DIR}")
TRAINED_CONTROLNET_PATH = Path(checkpoints[-1])

required_patterns = {
    "Qwen-Image transformer": str(QWEN_ROOT / "transformer/diffusion_pytorch_model*.safetensors"),
    "Qwen-Image text_encoder": str(QWEN_ROOT / "text_encoder/model*.safetensors"),
    "Qwen-Image vae": str(QWEN_ROOT / "vae/diffusion_pytorch_model.safetensors"),
    "Qwen-Image tokenizer": str(QWEN_ROOT / "tokenizer/*"),
}

missing = [name for name, pattern in required_patterns.items() if len(glob.glob(pattern)) == 0]
for path_name, path_value in [
    ("trained controlnet checkpoint", TRAINED_CONTROLNET_PATH),
    ("input image", INPUT_IMAGE_PATH),
    ("inpaint mask", MASK_IMAGE_PATH),
]:
    if not Path(path_value).exists():
        missing.append(f"{path_name}: {path_value}")

if missing:
    raise FileNotFoundError("Missing required files:\n- " + "\n- ".join(missing))

if torch.cuda.is_available():
    torch.cuda.set_device(CUDA_DEVICE_INDEX)
    device = f"cuda:{CUDA_DEVICE_INDEX}"
    torch_dtype = torch.bfloat16
else:
    device = "cpu"
    torch_dtype = torch.float32

print(f"[device] {device}")
print(f"[sample] index={VAL_SAMPLE_INDEX} / total={len(val_records)}")
print(f"[prompt] {PROMPT}")
print(f"[input] {INPUT_IMAGE_PATH}")
print(f"[mask] {MASK_IMAGE_PATH}")
print(f"[target] {TARGET_IMAGE_PATH}")
print(f"[controlnet ckpt] {TRAINED_CONTROLNET_PATH}")


In [ ]:
def _model_config_from_glob(local_glob: str, label: str) -> ModelConfig:
    paths = sorted(glob.glob(local_glob))
    if not paths:
        raise FileNotFoundError(f"Missing local weights for {label}: {local_glob}")
    path_value = paths if len(paths) > 1 else paths[0]
    print(f"[model] {label}: {path_value}")
    return ModelConfig(path=path_value)


def _dir_config(path: Path, label: str) -> ModelConfig:
    if not path.exists():
        raise FileNotFoundError(f"Missing local directory for {label}: {path}")
    print(f"[aux] {label}: {path}")
    return ModelConfig(path=str(path))


pipe = QwenImagePipeline.from_pretrained(
    torch_dtype=torch_dtype,
    device=device,
    model_configs=[
        _model_config_from_glob(
            str(QWEN_ROOT / "transformer/diffusion_pytorch_model*.safetensors"),
            "Qwen-Image transformer",
        ),
        _model_config_from_glob(
            str(QWEN_ROOT / "text_encoder/model*.safetensors"),
            "Qwen-Image text encoder",
        ),
        _model_config_from_glob(
            str(QWEN_ROOT / "vae/diffusion_pytorch_model.safetensors"),
            "Qwen-Image vae",
        ),
        ModelConfig(path=str(TRAINED_CONTROLNET_PATH)),
    ],
    tokenizer_config=_dir_config(QWEN_ROOT / "tokenizer", "Qwen-Image tokenizer"),
    processor_config=None,
)


In [ ]:
def resize_by_max_long_side(image, max_long_side, resample):
    width, height = image.size
    long_side = max(width, height)
    if long_side <= max_long_side:
        return image
    scale = max_long_side / long_side
    new_size = (int(round(width * scale)), int(round(height * scale)))
    return image.resize(new_size, resample)


input_image = Image.open(INPUT_IMAGE_PATH).convert("RGB")
inpaint_mask = Image.open(MASK_IMAGE_PATH).convert("RGB")
target_image = Image.open(TARGET_IMAGE_PATH).convert("RGB") if TARGET_IMAGE_PATH.exists() else None

input_image = resize_by_max_long_side(input_image, MAX_LONG_SIDE, Image.LANCZOS)
if inpaint_mask.size != input_image.size:
    inpaint_mask = inpaint_mask.resize(input_image.size, Image.NEAREST)
if target_image is not None and target_image.size != input_image.size:
    target_image = target_image.resize(input_image.size, Image.LANCZOS)

generated_result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    input_image=input_image,
    cfg_scale=CFG_SCALE,
    seed=SEED,
    blockwise_controlnet_inputs=[ControlNetInput(image=input_image, inpaint_mask=inpaint_mask)],
    progress_bar_cmd="notebook",
    num_inference_steps=NUM_INFERENCE_STEPS,
)

output_dir = BUNDLE_ROOT / "output"
output_dir.mkdir(parents=True, exist_ok=True)
output_name = f"val_{VAL_SAMPLE_INDEX:05d}_{TRAINED_CONTROLNET_PATH.stem}_local.png"
output_path = output_dir / output_name
generated_result.save(output_path)

print(f"[saved] {output_path}")
display(generated_result)
